# 03 — Random vs. Smart Pruning (THE replication notebook)

This notebook reproduces the **empirical foundation of the thesis**:

| Condition | Q1: "What color is the car?" | Q2: "What is the weather?" | Q3: "Any people on the street?" |
|---|---|---|---|
| 🟢 Baseline (100% tokens) | ✅ correct | ✅ correct | ✅ correct |
| 🔴 Random (50% tokens) | ✅ correct | ✅ correct | ✅ correct |
| 🔵 Smart (50%, **WRONG location**, post-decoder) | ❌ gibberish | ❌ gibberish | ❌ gibberish |

**Two findings:**
1. **Random pruning works.** The LLM decoder is robust to 50% visual-token removal → massive redundancy.
2. **The smart failure was an implementation bug.** Scores were computed on post-decoder `last_hidden_state` (fully contextualized by the LLM). Scoring must happen at the **connector output (pre-decoder)**.

**Root cause (the wrong-location bug):** `outputs = model.model(**inputs, output_hidden_states=True)` → `outputs.last_hidden_state` is the *output* of the entire decoder. Every token there has been transformed by self-attention with all other tokens through all layers — there is no clean visual-token signal left to score, and pruning that corrupted signal destroys sequence structure → gibberish.

**Correct location (pre-decoder):** prune at `connector(vision_model(pixel_values).last_hidden_state)` — before the LLM decoder — exactly what `tgtp.models.smolvlm_wrapper` implements.

In [ ]:
!pip -q install torch transformers accelerate Pillow requests
import sys, os, torch
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import requests
from PIL import Image
from io import BytesIO
from transformers import AutoProcessor, AutoModelForImageTextToText
from tgtp.models.smolvlm_wrapper import SmolVLMPruningWrapper

MODEL = "HuggingFaceTB/SmolVLM-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
processor = AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True).to(device)
model.eval()
wrapper = SmolVLMPruningWrapper(model)

# SmolVLM at 384px: 81 visual tokens enter the LLM decoder (verified from config).
N_VIS = 81

In [ ]:
url = "https://images.unsplash.com/photo-1502877338535-766e1452684a?w=600"
img = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB").resize((384, 384))
img

In [ ]:
import torch.nn.functional as F

def greedy_decode_embeds(seq, max_new_tokens=12):
    """Greedy-argmax decode on an embedding sequence; returns generated token ids."""
    gen = []
    for _ in range(max_new_tokens):
        out = model.model.text_model(inputs_embeds=seq)
        nxt = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        gen.append(nxt)
        seq = torch.cat([seq, model.model.text_model.get_input_embeddings()(nxt)], dim=1)
    return torch.cat(gen, dim=1)

def run(method, keep_ratio=0.5, wrong_location=False):
    """Run the 3 questions under a given condition; returns list of answer strings."""
    answers = []
    for q in ["What color is the car?", "What is the weather?", "Any people on the street?"]:
        pv = processor(images=img, return_tensors="pt")["pixel_values"].to(device)
        enc = processor(text=q, return_tensors="pt")
        input_ids, mask = enc["input_ids"].to(device), enc["attention_mask"].to(device)
        with torch.no_grad():
            if method == "baseline":
                out = model.generate(pixel_values=pv, input_ids=input_ids, attention_mask=mask, max_new_tokens=12)
                gen = out[0, input_ids.shape[1]:]
            elif wrong_location:
                # ⚠️ WRONG LOCATION — reproduced for demonstration only.
                # Score the FULL model output (post-decoder). Every token is fully
                # contextualized here; pruning this corrupted representation
                # destroys the sequence structure -> gibberish (the original bug).
                full = model.model(pixel_values=pv, input_ids=input_ids, attention_mask=mask, output_hidden_states=True)
                fused = full.last_hidden_state                    # (1, N_vis + text, D)
                visual_part = fused[:, :N_VIS]
                text_part = fused[:, N_VIS:]
                text_mean = text_part.mean(dim=1, keepdim=True)
                sim = F.cosine_similarity(visual_part, text_mean.expand_as(visual_part), dim=-1)
                k = N_VIS // 2
                keep = torch.topk(sim, k).indices.sort().values
                pruned = fused[:, keep, :]
                seq = torch.cat([pruned, text_part], dim=1)
                gen = greedy_decode_embeds(seq)
            else:
                gen_ids, _, _ = wrapper.generate_with_pruning(
                    pv, input_ids, mask, keep_ratio=keep_ratio,
                    scoring_method=("random" if method == "random" else "cross_attention"),
                    max_new_tokens=12)
                gen = gen_ids[0, input_ids.shape[1]:]
        answers.append(processor.decode(gen, skip_special_tokens=True))
    return answers

In [ ]:
results = {
    "🟢 baseline (100%)": run("baseline"),
    "🔴 random (50%)": run("random", 0.5),
    "🔵 smart, CORRECT location (50%)": run("smart", 0.5),
    "⚠️ smart, WRONG location (post-decoder, 50%)": run("smart", 0.5, wrong_location=True),
}
for cond, ans in results.items():
    print(f"{cond}:")
    for q, a in zip(["Q1 color", "Q2 weather", "Q3 people"], ans):
        print(f"   {q:12s} -> {a[:70]}")
    print()

**Expected outcome:**
- Baseline, random 50%, and smart-50% (correct location) all answer correctly.
- Smart-50% at the *wrong location* (post-decoder) degrades — the exact bug documented in the Root cause section above.

The wrapper also hard-blocks post-decoder pruning so nobody can repeat the bug in the experiments:

In [ ]:
# D6 — location verification: the wrapper rejects post-decoder pruning loudly.
try:
    SmolVLMPruningWrapper.prune_from_last_hidden_state(torch.randn(1, 90, 2048))
except RuntimeError as e:
    print("GUARD ACTIVE:", str(e)[:110], "...")

In [ ]:
# Root cause summary (wrong-location bug vs. correct pre-decoder location)
#
# WRONG:
#   outputs = model.model(**inputs, output_hidden_states=True)
#   fused   = outputs.last_hidden_state        # FULLY CONTEXTUALIZED (post-decoder)
#   scores  = cosine(fused_visual, fused_text) # meaningless -> gibberish after pruning
#
# CORRECT (implemented in tgtp.models.smolvlm_wrapper):
#   visual = model.model.connector(model.model.vision_model(pixel_values).last_hidden_state)
#   text   = model.model.text_model.get_input_embeddings()(input_ids)
#   scores = cross_attention(visual, text)     # PRE-DECODER -> meaningful
#   keep   = topk(scores, k); prune; feed decoder
print("See src/tgtp/models/smolvlm_wrapper.py for the correct implementation.")